# Detektor Fashionpedia — YOLOv8 detect di Kaggle

Melatih **detektor** yang mencari kotak setiap pakaian dalam satu foto, lalu memakai classifier 18
kelas yang sudah ada untuk menamai isi tiap kotak (`docs/adr/0009`). Detektor ini hanya menjawab
*di mana* dan lima Category: Tops, Bottoms, Outerwear, Shoes, Accessories.

**Sebelum menjalankan:** isi `REPO_REF` di langkah 2 dengan branch yang memuat
`ml_workspace/scripts/fashionpedia_to_yolo.py`. Notebook ini meng-clone repo, jadi tanpa itu langkah
2 berhenti dengan pesan yang jelas.

### Setelan sesi

1. Panel kanan → **Accelerator → GPU T4** (atau P100).
2. Panel kanan → **Internet → On**: notebook mengunduh 3,48 GB parquet Fashionpedia dari HuggingFace
   dan meng-install `datasets` + `ultralytics`.
3. Tidak perlu *Add Input* apa pun — datanya diunduh dari HuggingFace.

### Yang dihasilkan

`garment_detector.tflite` (model deteksi), `report.md` (mAP50, mAP50-95, AP per Category), CSV/PNG
confusion matrix, dan `fashionpedia_yolo.json` (bukti isi dataset yang dipakai). Zip-nya diunduh dari
`/kaggle/working`.

### Lisensi

Anotasi Fashionpedia CC BY 4.0; fotonya mengikuti terms situs asalnya (Flickr, Unsplash, Burst,
Pexels, …). Bobot hasil latihan Ultralytics = AGPL-3.0 (`docs/adr/0006`), jadi build yang memuatnya
adalah artefak tugas, bukan untuk diedarkan (`docs/adr/0002`, `docs/adr/0003`).

## 1. Runtime dan GPU

Sel ini memastikan GPU terlihat dan versi ultralytics dicetak. Tanpa GPU, latihan 43.622 foto pada
640 px tidak akan selesai dalam batas sesi — dan itu akan dikatakan di sini, bukan dua jam kemudian.

In [ ]:
# Langkah 1/8 — cek runtime.
!pip install -q ultralytics "datasets>=3" "huggingface_hub>=0.23"

import sys
import torch
import ultralytics

print(f"python        {sys.version.split()[0]}")
print(f"torch         {torch.__version__}")
print(f"ultralytics   {ultralytics.__version__}")
print(f"cuda          {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"gpu           {torch.cuda.get_device_name(0)}")
else:
    print("              panel kanan -> Accelerator -> GPU T4, lalu jalankan ulang dari sel ini")

## 2. Ambil konverter dari repo

Detektor dilatih pada pohon YOLO yang dibangun `fashionpedia_to_yolo.py`; logika pemetaan 46
kategori Fashionpedia ke lima Category hanya ada di satu tempat, yaitu berkas itu. Notebook ini
karena itu meng-clone repo alih-alih menyalin ulang logikanya.

In [ ]:
# Langkah 2/8 — clone repo pada ref yang memuat konverter.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/kmilaz/gayaku.git"
REPO_REF = "main"          # ganti ke branch PR kalau konverternya belum di-merge
REPO_DIR = Path("/kaggle/temp/gayaku")


def run(command, **kwargs):
    print("+", " ".join(str(part) for part in command))
    completed = subprocess.run(command, text=True, **kwargs)
    if completed.returncode != 0:
        raise SystemExit(f"perintah gagal ({completed.returncode}): {' '.join(map(str, command))}")
    return completed


if not (REPO_DIR / ".git").is_dir():
    run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)])
CONVERTER = REPO_DIR / "ml_workspace/scripts/fashionpedia_to_yolo.py"
if not CONVERTER.is_file():
    raise SystemExit(
        f"{CONVERTER} tidak ada di ref '{REPO_REF}'.\n"
        "  Push branch yang memuat konverter, lalu set REPO_REF ke nama branch itu."
    )
print(f"konverter    {CONVERTER}")

## 3. Unduh Fashionpedia dan bangun pohon YOLO

Delapan berkas parquet (3,48 GB). Konversi memetakan 27 kategori item ke lima Category dan
**membuang 19 kategori bagian pakaian** — tanpa itu detektor belajar membingkai lengan baju.
`val` tetap split validasi resmi; 2.000 foto test diambil dari train supaya angka uji tidak pernah
memengaruhi pemilihan epoch.

Hasilnya di `/kaggle/temp` (tidak ikut tersimpan); manifest-nya disalin ke `/kaggle/working/artifacts`.

In [ ]:
# Langkah 3/8 — unduh parquet dan konversi ke pohon YOLO.
import subprocess
import sys
from pathlib import Path

from huggingface_hub import snapshot_download

HF_DIR = Path("/kaggle/temp/fashionpedia_hf")
YOLO_DIR = Path("/kaggle/temp/fashionpedia_yolo")
ARTIFACTS = Path("/kaggle/working/artifacts")
ARTIFACTS.mkdir(parents=True, exist_ok=True)

if not list((HF_DIR / "data").glob("*.parquet")):
    snapshot_download(
        repo_id="detection-datasets/fashionpedia",
        repo_type="dataset",
        local_dir=str(HF_DIR),
        allow_patterns=["data/*.parquet", "dataset_infos.json"],
    )
print(f"parquet       {len(list((HF_DIR / 'data').glob('*.parquet')))} berkas di {HF_DIR}")

if not (YOLO_DIR / "manifest.json").is_file():
    run(
        [sys.executable, str(CONVERTER), "--hf-dir", str(HF_DIR), "--out", str(YOLO_DIR), "--force"],
        cwd=str(REPO_DIR),
    )
print(f"pohon YOLO    {YOLO_DIR}")

## 4. Periksa pohon sebelum menghabiskan kuota GPU

Tiga pemeriksaan, semuanya berhenti kalau gagal: jumlah foto masuk akal (43 ribu lebih), `data.yaml`
menyebut lima Category, dan setiap label berisi kelas 0–4 dengan koordinat ternormalisasi.

In [ ]:
# Langkah 4/8 — periksa pohon dan cetak ringkasannya.
import json
from pathlib import Path

MANIFEST = json.loads((YOLO_DIR / "manifest.json").read_text(encoding="utf-8"))
TRAIN = MANIFEST["splits"]["train"]
if TRAIN["images_written"] < 40000:
    raise SystemExit(f"hanya {TRAIN['images_written']} foto train; konversinya tidak lengkap")
if MANIFEST["categories"] != ["Tops", "Bottoms", "Outerwear", "Shoes", "Accessories"]:
    raise SystemExit(f"kategori tidak sesuai: {MANIFEST['categories']}")

labels = sorted((YOLO_DIR / "labels/train").glob("*.txt"))[:200]
if not labels:
    raise SystemExit("tidak ada label untuk diperiksa")
for path in labels:
    for line in path.read_text(encoding="utf-8").splitlines():
        cls, cx, cy, w, h = line.split()
        if not 0 <= int(cls) <= 4 or not all(0 <= float(v) <= 1 for v in (cx, cy, w, h)):
            raise SystemExit(f"{path.name}: baris label tidak valid: {line}")

print(f"{'split':<8}{'foto':>8}{'box':>9}  per Category")
for split, stats in MANIFEST["splits"].items():
    print(f"{split:<8}{stats['images_written']:>8}{stats['boxes_written']:>9}  {stats['per_category']}")
print(f"box bagian-pakaian dibuang: {MANIFEST['splits']['train']['dropped_parts']}")
(ARTIFACTS / "fashionpedia_yolo.json").write_text(json.dumps(MANIFEST, indent=2), encoding="utf-8")

## 5. Latih detektor

`yolov8n.pt` (deteksi, 640 px). Kenopnya di awal sel: `EPOCHS` 40, `BATCH` 16, `PATIENCE` 10.
Pada T4, 43.622 foto ±7 menit per epoch, jadi 40 epoch ±5 jam — masih di dalam batas 12 jam sesi,
tapi jangan menjalankannya dua kali.

Deteksi bukan klasifikasi: outputnya kotak, jadi yang diukur nanti mAP, bukan top-1. Detektor ini
hanya lima Category; nama spesifiknya datang dari classifier 18 kelas yang sudah dilatih.

In [ ]:
# Langkah 5/8 — latih yolov8n deteksi.
from pathlib import Path

from ultralytics import YOLO

EPOCHS = 40
BATCH = 16
IMGSZ = 640
PATIENCE = 10
RUN_NAME = "yolov8n-det-fashionpedia-640"
RUNS_DIR = Path("/kaggle/temp/runs")
RUNS_DIR.mkdir(parents=True, exist_ok=True)

model = YOLO("yolov8n.pt")
model.train(
    data=str(YOLO_DIR / "data.yaml"),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    device=0,
    workers=4,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    seed=42,
)
BEST = Path(model.trainer.save_dir) / "weights" / "best.pt"
print(f"bobot terbaik {BEST} ({BEST.stat().st_size / 1e6:.1f} MB)")
print(f"kelas model   {[model.names[i] for i in range(len(model.names))]}")

## 6. Evaluasi pada split `test`

`val` dipakai memilih epoch; `test` (2.000 foto dari train, tidak pernah dilihat model) yang
dilaporkan. Ultralytics memberi mAP50 dan mAP50-95; AP per Category menunjukkan mana yang paling
sering meleset — dugaan saya `Outerwear`, karena box mantel dan blazer saling tumpang tindih di foto
orang berpakaian.

In [ ]:
# Langkah 6/8 — evaluasi pada test dan simpan angkanya.
import csv
import numpy as np
from ultralytics import YOLO

trained = YOLO(str(BEST))
metrics = trained.val(
    data=str(YOLO_DIR / "data.yaml"),
    split="test",
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    plots=True,
    project=str(RUNS_DIR),
    name="val-test",
    exist_ok=True,
)

MAP50 = float(metrics.box.map50)
MAP5095 = float(metrics.box.map)
print(f"mAP50         {MAP50:.4f}")
print(f"mAP50-95      {MAP5095:.4f}")

names = metrics.names
rows = []
for index, name in names.items():
    rows.append((name, float(metrics.box.ap50[index]) if index < len(metrics.box.ap50) else float("nan")))
print(f"{'Category':<14}{'AP50':>8}")
for name, ap in rows:
    print(f"{name:<14}{ap:>8.4f}")

with (ARTIFACTS / "per_category_ap.csv").open("w", encoding="utf-8", newline="") as handle:
    writer = csv.writer(handle)
    writer.writerow(("category", "ap50"))
    for name, ap in rows:
        writer.writerow((name, f"{ap:.4f}"))

## 7. Ekspor ke `.tflite`

Untuk deteksi, graf ekspornya memuat decoding kotak; `format` yang benar mengikuti versi
ultralytics di sesi ini (`litert` pada versi baru, `tflite` pada yang lebih lama). Sel mencetak
bentuk tensor hasilnya apa adanya — integrasi aplikasi untuk deteksi belum dibangun, jadi bentuk itu
yang akan jadi patokan, bukan asumsi.

In [ ]:
# Langkah 7/8 — ekspor dan baca kontrak tensor.
import hashlib
from pathlib import Path

import numpy as np

TFLITE = ARTIFACTS / "garment_detector.tflite"
exported = None
for export_format in ("litert", "tflite"):
    try:
        exported = Path(trained.export(format=export_format, imgsz=IMGSZ, device="cpu"))
        print(f"diekspor dengan format={export_format}")
        break
    except Exception as exc:
        print(f"format={export_format} gagal: {type(exc).__name__}: {exc}")
if exported is None:
    raise SystemExit("ekspor TFLite gagal untuk kedua nama format")

import shutil

shutil.copy2(exported, TFLITE)
print(f"{TFLITE.name}  {TFLITE.stat().st_size:,} bytes")
print(f"sha256       {hashlib.sha256(TFLITE.read_bytes()).hexdigest()}")

from ai_edge_litert.interpreter import Interpreter

interpreter = Interpreter(model_path=str(TFLITE))
interpreter.allocate_tensors()
for tensor in interpreter.get_input_details():
    print(f"input        {tensor['name']} {tuple(tensor['shape'])} {np.dtype(tensor['dtype']).name}")
for tensor in interpreter.get_output_details():
    print(f"output       {tensor['name']} {tuple(tensor['shape'])} {np.dtype(tensor['dtype']).name}")

## 8. Tulis laporan dan bungkus

`report.md` menyimpan versi runtime, angka mAP, AP per Category, isi dataset, dan catatan lisensi.
Semua artefak dibungkus satu zip di `/kaggle/working`.

In [ ]:
# Langkah 8/8 — laporan dan zip.
import json
import zipfile
from datetime import datetime, timezone

REPORT = f"""# yolov8n detection on Fashionpedia — GPU run

| | |
| --- | --- |
| date (UTC) | {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M')} |
| notebook | `ml_workspace/yolo/kaggle_detector.ipynb` |
| repo ref | `{REPO_REF}` |
| dataset | `detection-datasets/fashionpedia` via `ml_workspace/scripts/fashionpedia_to_yolo.py` |
| photographs | {MANIFEST['splits']['train']['images_written']} train / {MANIFEST['splits']['val']['images_written']} val / {MANIFEST['splits']['test']['images_written']} test |
| boxes | {MANIFEST['splits']['train']['boxes_written']} train item boxes, {MANIFEST['splits']['train']['dropped_parts']} part boxes dropped |
| training | yolov8n detection, {EPOCHS} epochs, imgsz {IMGSZ}, batch {BATCH}, patience {PATIENCE} |
| GPU | {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'} |
| ultralytics / torch | {ultralytics.__version__} / {torch.__version__} |

## Measured on `test`

| metric | value |
| --- | --- |
| mAP50 | {MAP50:.4f} |
| mAP50-95 | {MAP5095:.4f} |

| Category | AP50 |
| --- | --- |
{chr(10).join(f'| {name} | {ap:.4f} |' for name, ap in rows)}

The detector answers *where* and which Category; the 18-class classifier names the crop inside each
box (`docs/adr/0009`). Detection accuracy here therefore does not replace the classifier's
per-class recall, and vice versa.

## Licence

Ultralytics-trained weights are AGPL-3.0 (`docs/adr/0006`). Fashionpedia annotations are CC BY 4.0;
its photographs keep the terms of the sites they came from. Coursework artifacts, not for
redistribution (`docs/adr/0003`).
"""
(ARTIFACTS / "report.md").write_text(REPORT, encoding="utf-8")
for png in sorted(Path(metrics.save_dir).glob("*.png")):
    shutil.copy2(png, ARTIFACTS / png.name)

ZIP = Path("/kaggle/working/gayaku_detector.zip")
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(ARTIFACTS.iterdir()):
        if path.is_file():
            archive.write(path, path.name)

print(f"ditulis       {ARTIFACTS / 'report.md'}")
print(f"zip           {ZIP} ({ZIP.stat().st_size:,} bytes)")
print()
print("Ringkasan untuk dilaporkan:")
print(f"  mAP50 test    {MAP50:.4f}")
print(f"  mAP50-95      {MAP5095:.4f}")
print(f"  foto train    {MANIFEST['splits']['train']['images_written']}")
print()
print("Selanjutnya: integrasi dua tahap (deteksi -> crop -> classifier 18 kelas) belum dibangun.")